# Class 14: Nothing Crashed
*Elements of Data Science — Honors*

**First thing:** save this notebook under a new name that includes your team's name.

Every cell in Part 2 runs without an error message. That is the problem. Keep the worksheet
next to you, and **write each prediction on the worksheet before you run the cell it is about.**

In [ ]:
from datascience import *
import numpy as np
# import for plotting
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
# Fix for datascience plots
import collections as collections
import collections.abc as abc
collections.Iterable = abc.Iterable

---
## The Data

Everything below is **invented**. Riverton is not a real city, Riverton Transit is not a real
bus system, and the numbers came out of a random number generator. That is deliberate: when
you made the data up yourself, you can know for certain what the right answers are.

The table holds every bus trip Riverton Transit ran in 2025 on its four lines — 620 trips. It
is sorted by date.

- `date`
- `line` — Blue, Red, Green, or Gold
- `minutes late` — how far behind schedule the bus arrived. A **negative** number means the
  bus was early.

In [ ]:
trips = Table.read_table('data/riverton_trips.csv')
trips.show(5)

---
## Part 2. The Intern's Notebook

A summer intern wrote the analysis below and turned in the report printed on your worksheet.
The intern's code is copied here exactly. Run it as written, then check each claim.

### Exhibit A — Claim 1

**A.1** Prediction on the worksheet first.

In [ ]:
# Intern's code
late_trips = trips.where('minutes late', are.above(5))
late_trips.group('line')

**A.2** The count above is real. "Worst" needs something to compare it to: how many trips
each line ran **in total**. Run this.

In [ ]:
all_trips = trips.group('line')
all_trips

**A.3** On the worksheet, work out Blue's fraction late **by hand**, from the two tables
above. Then let Python do all four lines at once.

Both tables list the lines in the same order (alphabetical), so dividing one `count` column
by the other divides each line's late trips by *that same line's* total trips. Here are the
two columns you will divide:

In [ ]:
print('late trips: ', late_trips.group('line').column('count'))
print('all trips:  ', all_trips.column('count'))

Fill in the bottom of the fraction. Check that Blue matches your hand calculation.

In [ ]:
late_counts = late_trips.group('line')

fraction_late = late_counts.column('count') / ...

all_trips.with_column('fraction late', fraction_late)

### Exhibit B — Claim 2, the sample

**B.1** Prediction on the worksheet first.

In [ ]:
# Intern's code
sample = trips.take(np.arange(80))
observed = np.count_nonzero(sample.column('minutes late') <= 5)
observed

**B.2** Look at the first and last dates in the intern's sample.

In [ ]:
dates = sample.column('date')
print('first trip:', dates.item(0))
print('last trip: ', dates.item(79))

**B.3** Now draw 80 trips **at random**, the way Class 13 taught you. `trips.sample(80)` picks
80 rows at random — but it has a setting you have to decide on:

- `with_replacement=True` — after a trip is picked, it goes back in the pile and **can be
  picked again**. (This is what `sample` does if you say nothing — like the marbles in Class 13.)
- `with_replacement=False` — each trip can be picked **at most once**.

Riverton ran each of these trips once. Should the same trip be allowed to appear in your
sample twice? Replace the `...` with `True` or `False`, and write your reason on the worksheet.

In [ ]:
sample = trips.sample(80, with_replacement=...)
observed = np.count_nonzero(sample.column('minutes late') <= 5)
observed

### Exhibit C — Claim 2, the test

The agency's claim: each trip has an 80% chance of arriving within 5 minutes of schedule. The
intern simulates 80 trips under that claim, ten thousand times, and asks how often the
simulation produces as few on-time trips as the sample did.

**C.1** First, one simulated sample of 80 trips, so you can see what the intern's long line of
code is doing. Run this cell a few times.

In [ ]:
claim = Table().with_columns('Outcome', make_array('on time', 'late'),
                             'Chance',  make_array(0.8, 0.2))

claim.sample_from_distribution('Chance', 80)

The `Chance sample` column says how many of the 80 simulated trips landed in each row. The
intern only wants the **on-time** count — the first number in that column:

In [ ]:
claim.sample_from_distribution('Chance', 80).column('Chance sample').item(0)

That one number is what the intern calls `one`. 

**C.2** Now do the trace on the worksheet (question 7) **before** you run the intern's code.
Note that `observed` is whatever **your** sample in B.3 gave — the intern's test runs on your
sample, not the intern's.

In [ ]:
# Intern's code
simulated = make_array()
for i in np.arange(10000):
    one = claim.sample_from_distribution('Chance', 80).column('Chance sample').item(0)
simulated = np.append(simulated, one)

p_value = np.count_nonzero(simulated <= observed) / 10000
p_value

**C.3** Our rule: every loop runs at 10 repetitions before it runs at 10,000. Run the intern's
loop at 10 and look at what comes out.

In [ ]:
simulated = make_array()
for i in np.arange(10):
    one = claim.sample_from_distribution('Chance', 80).column('Chance sample').item(0)
simulated = np.append(simulated, one)

simulated

**C.4** One line is in the wrong place. Below, it has been removed — put it back *inside*
the loop (indented, like the line above it). Check at 10, then change the 10 to 10000.

In [ ]:
simulated = make_array()
for i in np.arange(10):
    one = claim.sample_from_distribution('Chance', 80).column('Chance sample').item(0)
    ...

simulated

In [ ]:
Table().with_column('on-time trips out of 80', simulated).hist(bins=np.arange(50, 76, 1))

p_value = np.count_nonzero(simulated <= observed) / len(simulated)
p_value

Record your team's `observed` and `p_value` on the worksheet, then go on to Part 3 there.